# Нейросеть на PyTorch (Airline Passenger Satisfaction)

Данные **не бинаризуются**: числовые признаки масштабируются, текстовые заменяются на `+1` / `−1`. Сеть обучается на всех объектах с настоящими метками, на выходе — вероятность `True`.

1. Загрузка и подготовка данных
2. Обучение сети (метрики по эпохам)
3. Кривые обучения
4. Итоговые метрики
5. Прогноз для `test.csv` → `submission.csv`

In [1]:
import copy, os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

CSV = "train.csv"          # обучающий файл (разделитель ",")
PRED_CSV = "test.csv"      # файл для прогноза (без столбца satisfaction)
TARGET = "satisfaction"
ID_NAME = "id"             # имя столбца с номерами (регистр не важен)
ID_START = 699635          # если в test.csv нет столбца id - нумерация с этого числа
OUT = "out"
ORDER = {"Class": ["Eco", "Eco Plus", "Business"]}   # порядок значений для текстовых признаков, у которых больше двух значений

VAL_SIZE = 0.2             # доля данных для проверки
SEED = 42
HIDDEN = [64, 32]          # скрытые слои сети
EPOCHS = 30
BATCH = 1024
LR = 1e-3

os.makedirs(OUT, exist_ok=True)
torch.manual_seed(SEED)
np.random.seed(SEED)

## 1. Данные
* столбец `id` не используется как признак;
* числовые признаки (возраст, дальность, задержки, оценки 0–5): пропуски → медиана, затем `(x − среднее) / стандартное отклонение`;
* текстовые признаки: два значения → `−1` и `+1` (пол, тип клиента, тип поездки); если значений три, они идут по порядку из `ORDER`: `Eco = −1`, `Eco Plus = 0`, `Business = +1`;
* среднее, медиана и std считаются по обучающему файлу и так же применяются к `test.csv`.

In [2]:
train = pd.read_csv(CSV, sep=",")
train.columns = train.columns.str.strip()
train = train.drop(columns=[c for c in train.columns if c.lower() == ID_NAME or c.startswith("Unnamed")])

# целевой признак -> 0/1
y = train[TARGET].astype(str).str.strip().str.lower().map({"true": 1, "false": 0, "1": 1, "0": 0})
assert y.notna().all(), "неизвестные значения в целевом признаке"
y = y.to_numpy(dtype=np.float32)

features = [c for c in train.columns if c != TARGET]
num_cols = [c for c in features if pd.api.types.is_numeric_dtype(train[c])]
cat_cols = [c for c in features if c not in num_cols]

# параметры масштабирования (по обучающим данным)
median = train[num_cols].median()
mean = train[num_cols].fillna(median).mean()
std = train[num_cols].fillna(median).std().replace(0, 1)

# текстовые признаки: каждому значению - число от -1 до +1 (два значения: -1 и +1, три: -1, 0, +1)
code = {}
for c in cat_cols:
    values = ORDER.get(c) or sorted(train[c].dropna().unique())
    assert len(values) == 2 or c in ORDER, f"в столбце {c} больше двух значений, добавьте его в ORDER: {values}"
    assert set(train[c].dropna().unique()) <= set(values), f"в ORDER для {c} не хватает значений: {sorted(train[c].dropna().unique())}"
    code[c] = dict(zip(values, np.linspace(-1, 1, len(values)).tolist()))
print("Кодирование текстовых признаков:", code)


def prepare(df):
    """Таблица -> массив для сети (числа масштабируются, текст -> числа от -1 до +1)."""
    num = (df[num_cols].fillna(median) - mean) / std
    cat = pd.DataFrame({c: df[c].map(code[c]).fillna(0.0) for c in cat_cols}, index=df.index)   # неизвестное значение -> 0
    return pd.concat([num, cat], axis=1)[features].to_numpy(dtype=np.float32)


X = prepare(train)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=VAL_SIZE, stratify=y, random_state=SEED)
print("Признаков:", len(features), "| train:", X_train.shape, "| val:", X_val.shape, "| доля класса 1:", y.mean().round(3))

Кодирование текстовых признаков: {'Gender': {'Female': -1.0, 'Male': 1.0}, 'Customer Type': {'Loyal Customer': -1.0, 'disloyal Customer': 1.0}, 'Type of Travel': {'Business travel': -1.0, 'Personal Travel': 1.0}, 'Class': {'Eco': -1.0, 'Eco Plus': 0.0, 'Business': 1.0}}
Признаков: 21 | train: (559708, 21) | val: (139927, 21) | доля класса 1: 0.444


## 2. Обучение
Сеть: `вход → HIDDEN → 1`, активация ReLU, потеря `BCEWithLogitsLoss`, оптимизатор Adam. После каждой эпохи считаются метрики на `train` и `val`; в конце загружаются веса эпохи с наименьшей ошибкой на `val`.

In [3]:
# сеть
layers, n_in = [], X.shape[1]
for h in HIDDEN:
    layers += [nn.Linear(n_in, h), nn.ReLU()]
    n_in = h
model = nn.Sequential(*layers, nn.Linear(n_in, 1))
print(model)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

Xt, yt = torch.tensor(X_train), torch.tensor(y_train).unsqueeze(1)
Xv, yv = torch.tensor(X_val), torch.tensor(y_val).unsqueeze(1)
loader = DataLoader(TensorDataset(Xt, yt), batch_size=BATCH, shuffle=True)


def evaluate(Xs, ys):
    """Ошибка, доля верных ответов и AUC на выборке."""
    model.eval()
    with torch.no_grad():
        logits = model(Xs)
        loss = loss_fn(logits, ys).item()
        p = torch.sigmoid(logits).numpy().ravel()
    y_true = ys.numpy().ravel()
    return loss, accuracy_score(y_true, p > 0.5), roc_auc_score(y_true, p)


history, best_loss, best_state = [], float("inf"), None
for epoch in range(1, EPOCHS + 1):
    model.train()
    for xb, yb in loader:
        optimizer.zero_grad()
        loss_fn(model(xb), yb).backward()
        optimizer.step()

    tr_loss, tr_acc, tr_auc = evaluate(Xt, yt)
    va_loss, va_acc, va_auc = evaluate(Xv, yv)
    history.append(dict(epoch=epoch, train_loss=tr_loss, val_loss=va_loss, train_acc=tr_acc, val_acc=va_acc, train_auc=tr_auc, val_auc=va_auc))
    print(f"Эпоха {epoch:2d} | loss {tr_loss:.4f} / {va_loss:.4f} | acc {tr_acc:.4f} / {va_acc:.4f} | AUC {tr_auc:.4f} / {va_auc:.4f}   (train / val)")

    if va_loss < best_loss:
        best_loss, best_state, best_epoch = va_loss, copy.deepcopy(model.state_dict()), epoch

model.load_state_dict(best_state)
print("Лучшая эпоха:", best_epoch)
history = pd.DataFrame(history)

Sequential(
  (0): Linear(in_features=21, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=1, bias=True)
)


/home/danil/Maga/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:484: UserWarning: Found GPU0 NVIDIA GeForce GT 1030 which is of compute capability (CC) 6.1.
The following list shows the CCs this version of PyTorch was built for and the hardware CCs it supports:
- 7.5 which supports hardware CC >=7.5,<8.0
- 8.0 which supports hardware CC >=8.0,<9.0 except {8.7}
- 8.6 which supports hardware CC >=8.6,<9.0 except {8.7}
- 9.0 which supports hardware CC >=9.0,<10.0
- 10.0 which supports hardware CC >=10.0,<11.0 except {10.1}
- 12.0 which supports hardware CC >=12.0,<13.0
Your installed torch==2.14.0+cu130 does not include kernels for this GPU. Reinstall the same version against a CUDA build that does, e.g.:
  For CUDA 12.6 use pip install torch==2.14.0 --index-url https://download.pytorch.org/whl/cu126
  _warn_unsupported_code(d, device_cc, code_ccs)
/home/danil/Maga/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:602: UserWarning: 
NVIDIA GeForce GT 1030 with CUDA capa

Эпоха  1 | loss 0.2556 / 0.2583 | acc 0.9138 / 0.9125 | AUC 0.9520 / 0.9510   (train / val)
Эпоха  2 | loss 0.2489 / 0.2517 | acc 0.9176 / 0.9158 | AUC 0.9537 / 0.9526   (train / val)
Эпоха  3 | loss 0.2429 / 0.2461 | acc 0.9200 / 0.9182 | AUC 0.9546 / 0.9534   (train / val)
Эпоха  4 | loss 0.2410 / 0.2441 | acc 0.9210 / 0.9191 | AUC 0.9551 / 0.9540   (train / val)
Эпоха  5 | loss 0.2392 / 0.2428 | acc 0.9219 / 0.9201 | AUC 0.9556 / 0.9543   (train / val)
Эпоха  6 | loss 0.2370 / 0.2406 | acc 0.9220 / 0.9199 | AUC 0.9561 / 0.9547   (train / val)
Эпоха  7 | loss 0.2349 / 0.2388 | acc 0.9233 / 0.9209 | AUC 0.9565 / 0.9549   (train / val)
Эпоха  8 | loss 0.2352 / 0.2393 | acc 0.9229 / 0.9207 | AUC 0.9564 / 0.9548   (train / val)
Эпоха  9 | loss 0.2337 / 0.2378 | acc 0.9235 / 0.9210 | AUC 0.9568 / 0.9551   (train / val)
Эпоха 10 | loss 0.2348 / 0.2388 | acc 0.9234 / 0.9212 | AUC 0.9568 / 0.9550   (train / val)
Эпоха 11 | loss 0.2327 / 0.2375 | acc 0.9242 / 0.9216 | AUC 0.9571 / 0.9551   (t

## 3. Кривые обучения

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, name, title in zip(ax, ["loss", "acc", "auc"], ["Ошибка", "Доля верных ответов", "AUC"]):
    a.plot(history["epoch"], history[f"train_{name}"], label="train")
    a.plot(history["epoch"], history[f"val_{name}"], label="val")
    a.axvline(best_epoch, color="gray", ls=":", label="лучшая эпоха")
    a.set_title(title); a.set_xlabel("Эпоха"); a.grid(alpha=.3); a.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT, "training_curves.png"))
plt.show()

## 4. Итоговые метрики
Модель — с лучшей эпохи; порог 0,5.

In [ ]:
def predict_proba(X_array):
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(torch.tensor(X_array))).numpy().ravel()


def metrics(y_true, p):
    pred = (p > 0.5).astype(int)
    return {"Accuracy": accuracy_score(y_true, pred), "Precision": precision_score(y_true, pred),
            "Recall": recall_score(y_true, pred), "F1": f1_score(y_true, pred), "AUC": roc_auc_score(y_true, p)}


p_train, p_val = predict_proba(X_train), predict_proba(X_val)
print("Постоянный ответ (преобладающий класс) на val:", round(max(y_val.mean(), 1 - y_val.mean()), 4))
print("Confusion matrix (val):\n", confusion_matrix(y_val, p_val > 0.5))
pd.DataFrame({"train": metrics(y_train, p_train), "val": metrics(y_val, p_val)}).T.round(4)

## 5. Прогноз для test.csv
Файл обрабатывается так же, как обучающий (строки не удаляются). На выходе `submission.csv`: id и вероятность `True`.

In [ ]:
test = pd.read_csv(PRED_CSV, sep=",")
test.columns = test.columns.str.strip()

# номера: берём из файла, а если столбца нет - нумеруем с ID_START
id_col = next((c for c in test.columns if c.lower() == ID_NAME), None)
ids = test[id_col].to_numpy() if id_col else np.arange(ID_START, ID_START + len(test))

proba = predict_proba(prepare(test))
submission = pd.DataFrame({id_col or ID_NAME: ids, TARGET: proba})
submission.to_csv(os.path.join(OUT, "submission.csv"), index=False)
print(len(submission), "строк | id:", ids.min(), "-", ids.max(), "| средняя вероятность:", proba.mean().round(3))
submission.head()